# Example of solving a Steiner Tree/Forest Problem using HighsPy
This example demonstrates how to use the HighsPy library to solve a Steiner Tree Problem on a simple graph. We will create a graph with nodes and edges, define the terminals we want to connect, and then use the `SteinerProblem` class to find the optimal solution.

In [ ]:
import networkx as nx
from steinerpy import SteinerProblem

We make a simple example with four nodes and four edges. We want to connect A, B, and D. The optimal solution is AC, BC, and CD.

In [2]:
graph = nx.Graph()
edges = [("A", "C"), ("A", "D"), ("B", "C"), ("C", "D")]
weights = [1, 10, 1, 1]

for i, edge in enumerate(edges):
    graph.add_edge(edge[0], edge[1])
    graph.edges[edge][f"weight"] = weights[i]

We now create a `SteinerProblem` instance with the graph and the terminals we want to connect. We then call the `get_solution` method to solve the problem, specifying a time limit of 300 seconds.

In [3]:
solution = SteinerProblem(graph, [["A", "B", "D"]]).get_solution(time_limit=300)

INFO:root:Building the model.
INFO:root:Model built in 0.00 seconds.
INFO:root:Started with running the model...
INFO:root:Runtime: 0.01 seconds


Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 46 rows; 37 cols; 124 nonzeros; 21 integer variables (21 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [1e+00, 1e+01]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 1e+00]
Presolving model
30 rows, 23 cols, 78 nonzeros  0s
20 rows, 16 cols, 54 nonzeros  0s
11 rows, 7 cols, 29 nonzeros  0s
9 rows, 6 cols, 22 nonzeros  0s
5 rows, 5 cols, 12 nonzeros  0s
4 rows, 4 cols, 10 nonzeros  0s
Presolve reductions: rows 4(-42); columns 4(-33); nonzeros 10(-114) 
Objective function is integral with scale 1

Solving MIP model with:
   4 rows
   4 cols (4 binary, 0 integer, 0 implied int., 0 continuous, 0 domain fixed)
   10 nonzeros

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounded; X => User solution; Y 

We can now access the selected edges in the optimal solution, as well as the optimality gap, runtime, and objective value.

In [4]:
solution.selected_edges

[('A', 'C'), ('C', 'B'), ('C', 'D')]

In [5]:
print(f"Optimality gap: {solution.gap:.2f}")
print(f"Runtime: {solution.runtime:.2f} sec")
print(f"Objective: {solution.objective:.2f}")

Optimality gap: 0.00
Runtime: 0.01 sec
Objective: 3.00


In case you're interested in Steiner Forest Problems, you can define multiple sets of terminals to connect. For example, we can connect A and B in one set and C and D in another set.

In [7]:
solution = SteinerProblem(graph, [["A", "B"], ["C", "D"]]).get_solution(time_limit=300)


INFO:root:Building the model.
INFO:root:Model built in 0.00 seconds.
INFO:root:Started with running the model...
INFO:root:Runtime: 0.00 seconds


Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
MIP has 86 rows; 63 cols; 224 nonzeros; 31 integer variables (31 binary)
Coefficient ranges:
  Matrix  [1e+00, 1e+00]
  Cost    [1e+00, 1e+01]
  Bound   [1e+00, 1e+00]
  RHS     [1e+00, 1e+00]
Presolving model
50 rows, 37 cols, 121 nonzeros  0s
29 rows, 21 cols, 76 nonzeros  0s
22 rows, 15 cols, 55 nonzeros  0s
18 rows, 11 cols, 43 nonzeros  0s
14 rows, 10 cols, 32 nonzeros  0s
9 rows, 6 cols, 19 nonzeros  0s
6 rows, 4 cols, 13 nonzeros  0s
4 rows, 3 cols, 9 nonzeros  0s
1 rows, 2 cols, 2 nonzeros  0s
0 rows, 1 cols, 0 nonzeros  0s
0 rows, 0 cols, 0 nonzeros  0s
Presolve reductions: rows 0(-86); columns 0(-63); nonzeros 0(-224) - Reduced to empty
Presolve: Optimal

Src: B => Branching; C => Central rounding; F => Feasibility pump; H => Heuristic;
     I => Shifting; J => Feasibility jump; L => Sub-MIP; P => Empty MIP; R => Randomized rounding;
     S => Solve LP; T => Evaluate node; U => Unbounde

## Node-Weighted Steiner Tree (NWST)
In the **Node-Weighted Steiner Tree** problem, nodes have costs rather than (or in addition to) edges.  The goal is still to find a minimum-cost connected subgraph that spans a set of terminals, but now both node and edge costs contribute to the objective.

Internally, each node `v` is split into `v_in` and `v_out` connected by a directed edge whose cost equals the node weight.  The resulting edge-weighted directed graph is then solved with the standard formulation.

Use `NodeWeightedSteinerProblem(graph, terminal_groups, node_weights)` to model this variant.

In [ ]:
from steinerpy import NodeWeightedSteinerProblem

# Triangle graph: going via B is expensive (weight 10) so the optimal
# solution connects A and C directly.
G_nw = nx.Graph()
G_nw.add_edge('A', 'B')
G_nw.add_edge('B', 'C')
G_nw.add_edge('A', 'C')

node_weights = {'A': 1, 'B': 10, 'C': 1}
nw_solution = NodeWeightedSteinerProblem(G_nw, [['A', 'C']], node_weights).get_solution(time_limit=30)

print(f'Objective (node + edge costs): {nw_solution.objective:.2f}')  # 2.0
print(f'Selected edges: {nw_solution.selected_edges}')               # [(A, C)]
print(f'Selected nodes: {nw_solution.selected_nodes}')               # [A, C]

## Maximum-Weight Connected Subgraph (MWCS)
The **Maximum-Weight Connected Subgraph** problem asks for a connected subgraph whose total node weight is maximised.  Nodes may have positive *or negative* weights; the solver naturally excludes negative-weight nodes unless they are necessary connectors.

Use `MaxWeightConnectedSubgraph(graph, node_weights)` — a convenience subclass of `PrizeCollectingProblem` that automatically sets up the prize-collecting formulation.

In [ ]:
from steinerpy import MaxWeightConnectedSubgraph

G_mwcs = nx.Graph()
G_mwcs.add_edge('A', 'B', weight=0)
G_mwcs.add_edge('B', 'C', weight=0)
G_mwcs.add_edge('C', 'D', weight=0)

node_weights_mwcs = {'A': 2, 'B': 3, 'C': 1, 'D': -10}  # D is strongly negative
mwcs_solution = MaxWeightConnectedSubgraph(G_mwcs, node_weights_mwcs).get_solution(time_limit=30)

print(f'Selected nodes: {mwcs_solution.selected_nodes}')  # A, B, C (D excluded)
print(f'Total prize (= total node weight): {mwcs_solution.total_prize:.2f}')  # 6.0

## Degree-Constrained Steiner Tree
The **Degree-Constrained Steiner Tree** problem adds the restriction that no node in the solution may have more than `max_degree` incident edges.  One linear constraint per node is added to the MIP:

$$\sum_{e \ni v} x_e \leq D \quad \forall v$$

Use `DegreeConstrainedSteinerProblem(graph, terminal_groups, max_degree=D)`.

In [ ]:
from steinerpy import DegreeConstrainedSteinerProblem

G_deg = nx.Graph()
for u, v, w in [('A','B',1), ('B','C',1), ('C','D',1), ('A','C',1), ('B','D',1)]:
    G_deg.add_edge(u, v, weight=w)

deg_solution = DegreeConstrainedSteinerProblem(
    G_deg, [['A', 'D']], max_degree=2
).get_solution(time_limit=30)

print(f'Objective: {deg_solution.objective:.2f}')
print(f'Selected edges: {deg_solution.selected_edges}')
# Verify degree constraint
for node in G_deg.nodes():
    deg = sum(1 for e in deg_solution.selected_edges if node in e)
    print(f'  degree({node}) = {deg} (<= 2)')

## Budget-Constrained Steiner Tree
The **Budget-Constrained Steiner Tree** problem flips the optimisation direction: given a maximum total edge-cost budget, *maximise* the number of connected terminals.  A single budget constraint is added:

$$\sum_{e} c_e x_e \leq B$$

and the objective becomes minimising the number of unconnected terminals.

Use `BudgetConstrainedSteinerProblem(graph, terminal_groups, budget=B)`.

In [ ]:
from steinerpy import BudgetConstrainedSteinerProblem

G_bud = nx.Graph()
G_bud.add_edge('A', 'B', weight=1)
G_bud.add_edge('B', 'C', weight=1)
G_bud.add_edge('C', 'D', weight=1)

# With budget 2 we can afford at most 2 edges in this chain → connect 3 of 4 terminals
bud_solution = BudgetConstrainedSteinerProblem(
    G_bud, [['A', 'B', 'C', 'D']], budget=2.0
).get_solution(time_limit=30)

print(f'Connected: {bud_solution.connected_terminals} / {bud_solution.total_terminals}')
print(f'Selected edges: {bud_solution.selected_edges}')

## Directed Steiner Tree (Steiner Arborescence)
In the **Directed Steiner Tree** (also known as the *Steiner Arborescence Problem*), the graph is directed (`nx.DiGraph`) and a designated root node must reach every terminal via directed paths.  Only arcs in the direction they exist in the DiGraph are considered — no reverse arcs are added.

Use `DirectedSteinerProblem(digraph, root, terminals)`.

In [ ]:
from steinerpy import DirectedSteinerProblem

DG = nx.DiGraph()
DG.add_edge('A', 'B', weight=1)
DG.add_edge('B', 'C', weight=1)
DG.add_edge('A', 'C', weight=10)  # Direct but expensive

dir_solution = DirectedSteinerProblem(DG, root='A', terminals=['B', 'C']).get_solution(time_limit=30)

print(f'Objective (A→B→C): {dir_solution.objective:.2f}')  # 2.0
print(f'Selected arcs: {dir_solution.selected_edges}')     # [(A,B), (B,C)]
# Each arc is one-directional; reverse arcs (e.g. C→A) are not present
for u, v in dir_solution.selected_edges:
    print(f'  Arc ({u}→{v}) exists in DiGraph: {DG.has_edge(u, v)}')